# Hafta 5: Araç Fiyat Tahmini Projesi

Bu defterde gerçekçi bir araç veri seti oluşturup, lineer regresyon ile fiyat tahmini yapacağız.

**İçerik:**
- Veri oluşturma ve keşifsel veri analizi (EDA)
- Özellik mühendisliği (Feature Engineering)
- Model eğitimi ve değerlendirme
- Pipeline oluşturma
- Modeli kaydetme (joblib)

## 1. Gerekli Kütüphaneler

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `joblib` | Model kaydetme/yükleme |
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `seaborn` | İstatistiksel görselleştirme (Matplotlib üzerine kurulu) |
| `sklearn` | Makine öğrenmesi algoritmaları ve araçları |
| `warnings` | Uyarı mesajlarını yönetme |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Kütüphaneler başarıyla yüklendi!")

## 2. Gerçek Veri Setini Yükleme

**Automobile Dataset** — 205 araç kaydı, 26 özellik. Hedef değişken: `price`

**Kaynak:** [Kaggle - Car Price Prediction](https://www.kaggle.com/datasets/hellbuoy/car-price-prediction)

In [ ]:
# Kaggle Automobile (Car Price) veri setini yükle
url = "https://raw.githubusercontent.com/amankharwal/Website-data/master/CarPrice.csv"
df = pd.read_csv(url)

# CarName'den marka bilgisini çıkar
df['Marka'] = df['CarName'].apply(lambda x: x.split(' ')[0].lower())

# Bazı marka isimlerini düzelt (yazım hataları var)
marka_duzelt = {
    'maxda': 'mazda', 'porcshce': 'porsche', 'toyouta': 'toyota',
    'vokswagen': 'volkswagen', 'vw': 'volkswagen', 'alfa-romero': 'alfa-romeo'
}
df['Marka'] = df['Marka'].replace(marka_duzelt)

print(f"Veri seti boyutu: {df.shape}")
print(f"Sütunlar: {list(df.columns)}")
print(f"\nBenzersiz marka sayısı: {df['Marka'].nunique()}")
print(f"Markalar: {sorted(df['Marka'].unique())}")
print(f"\nFiyat aralığı: ${df['price'].min():,.0f} - ${df['price'].max():,.0f}")
df.head()

## 3. Keşifsel Veri Analizi (EDA)

### Temel İstatistikler

Verinin genel yapısını inceliyoruz: sütun tipleri, eksik değerler, temel istatistikler (ortalama, medyan, min, max). Bu bilgiler veri temizleme ve ön işleme adımlarını planlamak için gereklidir.

In [ ]:
# Temel istatistikler
print("Sayısal Değişkenlerin İstatistikleri:")
print("=" * 60)
df.describe().round(0)

### Veri Ön İşleme

Modeli eğitmeden önce veriyi temizliyoruz ve dönüştürüyoruz. Eksik değerleri doldurma, kategorik değişkenleri sayıya çevirme ve ölçeklendirme bu adımın parçasıdır.

In [ ]:
# Eksik değer kontrolü
print("Eksik Değer Sayıları:")
print("-" * 30)
print(df.isnull().sum())
print(f"\nToplam eksik değer: {df.isnull().sum().sum()}")

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Kategorik değişkenlerin dağılımı
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df['Marka'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='navy')
axes[0].set_title('Marka Dağılımı')
axes[0].set_xlabel('Marka')
axes[0].set_ylabel('Adet')
axes[0].tick_params(axis='x', rotation=45)

df['fueltype'].value_counts().plot(kind='bar', ax=axes[1], color='coral', edgecolor='darkred')
axes[1].set_title('Yakıt Tipi Dağılımı')
axes[1].set_xlabel('Yakıt Tipi')
axes[1].set_ylabel('Adet')

df['drivewheel'].value_counts().plot(kind='bar', ax=axes[2], color='mediumseagreen', edgecolor='darkgreen')
axes[2].set_title('Vites Tipi Dağılımı')
axes[2].set_xlabel('drivewheel')
axes[2].set_ylabel('Adet')

plt.tight_layout()
plt.show()

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
# Sayısal değişkenlerin dağılımları
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df['price'], bins=30, color='steelblue', edgecolor='navy', alpha=0.7)
axes[0, 0].set_title('Fiyat ($) Dağılımı')
axes[0, 0].set_xlabel('Fiyat ($)')

axes[0, 1].hist(df['citympg'], bins=30, color='coral', edgecolor='darkred', alpha=0.7)
axes[0, 1].set_title('Şehir İçi Yakıt (mpg) Dağılımı')
axes[0, 1].set_xlabel('citympg')

axes[1, 0].hist(df['horsepower'], bins=15, color='mediumseagreen', edgecolor='darkgreen', alpha=0.7)
axes[1, 0].set_title('Beygir Gücü Dağılımı')
axes[1, 0].set_xlabel('Yıl')

axes[1, 1].hist(df['enginesize'], bins=9, color='mediumpurple', edgecolor='indigo', alpha=0.7)
axes[1, 1].set_title('Motor Hacmi (cc) Dağılımı')
axes[1, 1].set_xlabel('Motor Hacmi (L)')

plt.suptitle('Sayısal Değişken Dağılımları', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Kutu Grafiği (Box Plot)

Kutu grafiği verinin dağılımını, medyanını, çeyrekliklerini ve uç değerlerini (outlier) gösterir. Gruplar arası karşılaştırma için idealdir.

In [ ]:
# Markaya göre fiyat kutusu grafiği
plt.figure(figsize=(12, 6))
marka_sirali = df.groupby('Marka')['price'].median().sort_values().index
sns.boxplot(data=df, x='Marka', y='price', order=marka_sirali, palette='viridis')
plt.title('Markaya Göre Fiyat ($) Dağılımı')
plt.xlabel('Marka')
plt.ylabel('Fiyat ($)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 4. Özellik Mühendisliği (Feature Engineering)

Kategorik değişkenleri sayısal hale dönüştürmemiz gerekiyor.

In [ ]:
# LabelEncoder ile kategorik değişkenleri dönüştürme
df_encoded = df.copy()

label_encoders = {}
kategorik_sutunlar = ['Marka', 'fueltype', 'carbody', 'drivewheel']

for col in kategorik_sutunlar:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print("\nDönüştürülmüş verinin ilk 5 satırı:")
df_encoded.head()

### Korelasyon Analizi ve Isı Haritası

Özellikler arasındaki ilişkiyi korelasyon matrisi ve ısı haritası ile görselleştiriyoruz. Yüksek korelasyonlu özellikler modelin en çok yararlanacağı özelliklerdir.

In [ ]:
# Korelasyon matrisi
plt.figure(figsize=(10, 8))
corr = df_encoded.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Korelasyon Matrisi')
plt.tight_layout()
plt.show()

print("\nFiyat ile korelasyonlar:")
print(corr['price'].sort_values(ascending=False).to_string())

## 5. Model Eğitimi

### Eğitim ve Test Setlerine Ayırma

Veriyi eğitim ve test olarak ikiye bölüyoruz. `stratify` parametresi, her iki sette de sınıf dağılımının aynı kalmasını sağlar. `random_state` ile tekrarlanabilir sonuçlar elde ediyoruz.

In [ ]:
# Özellik ve hedef ayrımı
X = df_encoded.drop('price', axis=1)
y = df_encoded['price']

# Eğitim/Test ayrımı (%80/%20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Eğitim seti: {X_train.shape[0]} örnek")
print(f"Test seti  : {X_test.shape[0]} örnek")

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Lineer Regresyon modeli
model = LinearRegression()
model.fit(X_train, y_train)

# Tahmin
y_pred = model.predict(X_test)

# Değerlendirme
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=" * 50)
print("  ARAÇ FİYAT TAHMİN MODELİ - SONUÇLAR")
print("=" * 50)
print(f"  MAE  : {mae:,.0f} TL")
print(f"  RMSE : {rmse:,.0f} TL")
print(f"  R²   : {r2:.4f}")
print("=" * 50)
print(f"\n  Ortalama fiyat: {y.mean():,.0f} TL")
print(f"  RMSE / Ort. Fiyat: %{(rmse / y.mean()) * 100:.1f}")

## 6. Görselleştirme: Gerçek vs Tahmin

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
# Gerçek vs Tahmin scatter plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sol: Scatter plot
axes[0].scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolors='navy', s=30)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='İdeal Çizgi')
axes[0].set_xlabel('Gerçek Fiyat ($)')
axes[0].set_ylabel('Tahmin Edilen Fiyat ($)')
axes[0].set_title('Gerçek vs Tahmin Edilen Fiyatlar')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Sağ: Hata dağılımı
hatalar = y_test - y_pred
axes[1].hist(hatalar, bins=30, color='coral', edgecolor='darkred', alpha=0.7)
axes[1].axvline(x=0, color='black', linewidth=2, linestyle='--')
axes[1].set_xlabel('Tahmin Hatası (TL)')
axes[1].set_ylabel('Frekans')
axes[1].set_title('Tahmin Hatası Dağılımı')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Modeli Kaydetme (joblib)

### Özellik Seçimi ve Mühendisliği

Model için kullanılacak özellikleri belirliyoruz. Doğru özellik seçimi model performansını doğrudan etkiler.

In [ ]:
# Modeli ve encoder'ları kaydetme
model_data = {
    'model': model,
    'label_encoders': label_encoders,
    'features': list(X.columns)
}

joblib.dump(model_data, 'arac_fiyat_modeli.pkl')
print("Model başarıyla 'arac_fiyat_modeli.pkl' dosyasına kaydedildi!")

# Doğrulama: Kaydedilen modeli yükleyip test edelim
yuklenen = joblib.load('arac_fiyat_modeli.pkl')
y_pred_kontrol = yuklenen['model'].predict(X_test)
print(f"Yüklenen model R² skoru: {r2_score(y_test, y_pred_kontrol):.4f}")
print("Doğrulama başarılı!")

## 8. sklearn Pipeline ile Model

Pipeline, veri ön işleme ve model eğitimini tek bir adımda yapmamızı sağlar.

In [ ]:
# Pipeline oluşturma: Ölçeklendirme + Lineer Regresyon
pipeline = Pipeline([
    ('scaler', StandardScaler()),      # Özellikleri standartlaştırma
    ('regressor', LinearRegression())    # Lineer Regresyon modeli
])

# Pipeline'ı eğitme
pipeline.fit(X_train, y_train)

# Tahmin ve değerlendirme
y_pred_pipe = pipeline.predict(X_test)

print("Pipeline Sonuçları:")
print("=" * 40)
print(f"  MAE  : {mean_absolute_error(y_test, y_pred_pipe):,.0f} TL")
print(f"  RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_pipe)):,.0f} TL")
print(f"  R²   : {r2_score(y_test, y_pred_pipe):.4f}")
print("=" * 40)
print("\nPipeline adımları:")
for name, step in pipeline.steps:
    print(f"  - {name}: {step.__class__.__name__}")

### Model Kaydetme

Eğitilmiş modeli diske kaydediyoruz. Böylece modeli tekrar eğitmeden doğrudan yükleyip tahmin yapabiliriz.

In [ ]:
# Pipeline'ı da kaydedelim
joblib.dump(pipeline, 'arac_fiyat_pipeline.pkl')
print("Pipeline 'arac_fiyat_pipeline.pkl' dosyasına kaydedildi!")

## 9. Örnek Tahmin

Kaydedilen modeli kullanarak yeni bir araç için fiyat tahmini yapalım.

In [ ]:
# Yeni araç verisi
yeni_arac = {
    'Marka': 'bmw',
    'horsepower': 2021,
    'citympg': 45000,
    'fueltype': 'Benzin',
    'drivewheel': 'Otomatik',
    'enginesize': 2.0
}

# Encoder ile dönüştürme
model_data = joblib.load('arac_fiyat_modeli.pkl')
yeni_df = pd.DataFrame([yeni_arac])

for col in ['Marka', 'fueltype', 'drivewheel']:
    yeni_df[col] = label_encoders[col].transform(yeni_df[col])

tahmin = model.predict(yeni_df)[0]

print("Araç Bilgileri:")
print("-" * 30)
for k, v in yeni_arac.items():
    print(f"  {k}: {v}")
print("-" * 30)
print(f"  Tahmini Fiyat: {tahmin:,.0f} TL")

## 10. Özet

Bu defterde:

1. Türkiye ikinci el araç piyasasına uygun **sentetik veri seti** oluşturduk
2. **Keşifsel Veri Analizi (EDA)** ile veriyi tanıdık
3. **LabelEncoder** ile kategorik değişkenleri dönüştürdük
4. **Lineer Regresyon** modeli eğitip değerlendirdik
5. Gerçek vs Tahmin grafiği ve hata analizi yaptık
6. Modeli **joblib** ile kaydettik
7. **sklearn Pipeline** kullanarak daha temiz bir iş akışı oluşturduk

Bir sonraki defterde bu modeli **Gradio** ile web arayüzüne dönüştüreceğiz!